# Deploy the Container Vulnerability Analysis Pipeline


This notebook deploys the current NVIDIA vulnerability-analysis workflow described in the README. The workflow is registered with NeMo Agent Toolkit (NAT) as the `cve_pipeline` graph and hosted as an HTTP service with `nat serve`.

The pipeline accepts one or more CVE/GHSA identifiers plus `image_name` and `image_tag`. If `filesystem_path` is omitted, the orchestrator pulls and extracts the container rootfs with `crane`, caches it, and writes per-agent Markdown reports plus final VEX JSON under `output_dir`.

**Cell classifications:** every code cell is classified in notebook metadata as `setup-only`, `executable`, or `illustrative`. Setup-only cells prepare repository or credential state; executable cells are expected to run as written in a configured environment; illustrative cells would show non-runnable templates. This deployment notebook currently contains no illustrative code cells.


# Getting Started

>[Prerequisites](#Prerequisites)  
>[Repository Setup](#Repository-Setup)  
>[Configure Environment](#Configure-Environment)  
>[Deploy with Docker Compose](#Deploy-with-Docker-Compose)  
>[Run the HTTP Workflow](#Run-the-HTTP-Workflow)  
>[Inspect Outputs](#Inspect-Outputs)  
>[API Reference](#API-Reference)  
>[Next Steps](#Next-Steps)  
>[Troubleshooting](#Troubleshooting)


## Prerequisites

- Docker and Docker Compose
- `NVIDIA_API_KEY` for NVIDIA OpenAI-compatible NIM inference
- `TAVILY_API_KEY` for web search

Both keys are required for a complete, uncached run. Provide them through the project-root `.env`, exported environment variables, or notebook/orchestrator secret injection; never commit real credentials. Compatible endpoint proxies can be selected with `NV_BASE_URL` and `TAVILY_API_BASE_URL`, but the key variables must still contain the credentials expected by those endpoints.

Run this notebook from top to bottom. It creates repository, environment, and deployment state used by later cells and by `2_Customize_CVE.ipynb`. After a kernel restart, rerun setup and producer cells rather than relying on files left in `/tmp`.

The current workflow does not require separate GHSA, NVD, SerpApi, SBOM, or source repository inputs for normal pipeline runs.


## Repository Setup

If you launched this notebook outside the repository, the setup cell clones a fallback checkout and detaches it at the reviewed release reference configured in the next cell. Set `VULN_ANALYSIS_REF` only when you intentionally want to test another commit or tag. If the notebook is already inside `vulnerability-analysis/deploy`, the cell preserves that local checkout and uses its current revision. For a reproducibility-sensitive run, set `VULN_ANALYSIS_EXPECTED_COMMIT` to the complete expected commit SHA; setup stops if the resolved checkout differs. The resolved SHA is always printed for the run record.


In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = "https://github.com/NVIDIA-AI-Blueprints/vulnerability-analysis.git"
REVIEWED_REPOSITORY_REF = "3.0.0"
REPOSITORY_REF = os.environ.get("VULN_ANALYSIS_REF", REVIEWED_REPOSITORY_REF)
EXPECTED_REPOSITORY_COMMIT = os.environ.get("VULN_ANALYSIS_EXPECTED_COMMIT", "").strip() or None


def locate_repo(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "vuln_analysis").is_dir():
            return candidate
    return None


try:
    launch_dir = Path.cwd()
except FileNotFoundError:
    launch_dir = Path.home()
    os.chdir(launch_dir)
    print(f"Kernel working directory was removed; continuing from {launch_dir}")

repo_root = locate_repo(launch_dir)
if repo_root is None:
    checkout = launch_dir / "vulnerability-analysis"
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--recurse-submodules", REPOSITORY_URL, str(checkout)],
            check=True,
        )
    elif not (checkout / ".git").exists():
        raise RuntimeError(f"Fallback checkout path exists but is not a Git repository: {checkout}")
    dirty = subprocess.run(
        ["git", "-C", str(checkout), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout
    if dirty:
        raise RuntimeError(f"Fallback checkout has local changes; preserve or remove them before continuing: {checkout}")
    subprocess.run(["git", "-C", str(checkout), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", REPOSITORY_REF], check=True)
    repo_root = checkout

subprocess.run(["git", "-C", str(repo_root), "submodule", "update", "--init", "--recursive"], check=True)
if not (repo_root / "aiq" / "pyproject.toml").is_file():
    raise RuntimeError("AI-Q submodule initialization failed: aiq/pyproject.toml is missing.")
os.chdir(repo_root)
resolved_ref = subprocess.run(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if EXPECTED_REPOSITORY_COMMIT and resolved_ref != EXPECTED_REPOSITORY_COMMIT:
    raise RuntimeError(
        "Resolved repository commit does not match VULN_ANALYSIS_EXPECTED_COMMIT: "
        f"{resolved_ref} != {EXPECTED_REPOSITORY_COMMIT}"
    )
print(f"Using repository: {repo_root} ({resolved_ref})")


In [ ]:
REPO_ROOT = repo_root
os.environ["REPO_ROOT"] = str(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")


## Configure Environment

The next cells ensure a project-root `.env` exists and offer masked prompts for the required API keys. The prompts are a convenience: press Enter to keep values already supplied by `.env` or environment/secret injection. Re-run the API-key cell whenever you intentionally need to update a local key. `NV_BASE_URL` is optional; leave it unset to use the configuration default.


In [ ]:
import shutil

env_path = REPO_ROOT / ".env"
if not env_path.exists():
    shutil.copyfile(REPO_ROOT / ".env.example", env_path)
    print(f"Created {env_path}")
else:
    print(f"Using existing {env_path}")

subprocess.run(
    ["bash", str(REPO_ROOT / "deploy" / "configure-host-identity.sh"), str(env_path)],
    check=True,
)


In [ ]:
import getpass


def read_dotenv_values(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('\"').strip("'")
    return values


def update_dotenv_values(path: Path, updates: dict[str, str]) -> None:
    lines = path.read_text(encoding="utf-8").splitlines() if path.exists() else []
    updated_lines = []
    updated_keys = set()
    for line in lines:
        candidate = line.strip()
        key = candidate.split("=", 1)[0].strip() if "=" in candidate else None
        if key in updates and not candidate.startswith("#"):
            updated_lines.append(f"{key}={updates[key]}")
            updated_keys.add(key)
        else:
            updated_lines.append(line)
    for key, value in updates.items():
        if key not in updated_keys:
            updated_lines.append(f"{key}={value}")
    path.write_text("\n".join(updated_lines) + "\n", encoding="utf-8")


api_key_names = ("NVIDIA_API_KEY", "TAVILY_API_KEY")
file_env = read_dotenv_values(env_path)
updates = {}
print("Enter new API keys below. Input is hidden; leave a value blank to keep the current key.")
for key in api_key_names:
    file_value = file_env.get(key, "")
    current_value = file_value or os.environ.get(key, "")
    has_current_value = bool(current_value and not current_value.startswith("your-"))
    suffix = " (press Enter to keep current value)" if has_current_value else ""
    value = getpass.getpass(f"{key}{suffix}: ").strip()
    if value:
        updates[key] = value
    elif file_value and not file_value.startswith("your-"):
        os.environ[key] = file_value

if updates:
    update_dotenv_values(env_path, updates)
    for key, value in updates.items():
        os.environ[key] = value
    print(f"Updated {len(updates)} API key(s) in {env_path}.")
else:
    print("No API keys were changed.")


In [ ]:
file_env = read_dotenv_values(env_path)
for key, value in file_env.items():
    os.environ.setdefault(key, value)

required = ["NVIDIA_API_KEY", "TAVILY_API_KEY"]
missing = [key for key in required if not os.environ.get(key) or os.environ[key].startswith("your-")]
if missing:
    raise RuntimeError(f"Fill in {', '.join(missing)} in {env_path}, then re-run this cell.")
else:
    print("Required API keys are present.")
print("NV_BASE_URL:", os.environ.get("NV_BASE_URL", "config/compose default"))


## Deploy with Docker Compose

Docker Compose builds the `vuln-analysis` image, starts the Jupyter container, starts the dedicated analysis API, and starts the nginx cache service used by the default container environment. The Dockerfile includes `uv`, `crane`, NAT, and the project package.


In [ ]:
%%bash
set -euo pipefail

COMPOSE=(bash deploy/compose.sh)

"${COMPOSE[@]}" build vuln-analysis
"${COMPOSE[@]}" up -d vuln-analysis vuln-analysis-api nginx-cache
"${COMPOSE[@]}" ps


## Run the HTTP Workflow

The `vuln-analysis-api` container runs `nat serve` as its main process. It listens on port `26466`, exposes Swagger docs at `/docs`, and accepts the same `PipelineInput` JSON used by `nat run`. Wait for `/ready` before submitting work; readiness requires the API, Dask scheduler, and at least one running worker. The local default allows one concurrent async job and starts one additional worker intended for cleanup, with a best-effort Dask memory limit of 40% of detected memory per worker. See `deploy/README.md` before increasing either resource setting.


In [ ]:
!curl --retry 30 --retry-delay 3 --retry-connrefused -fsS http://localhost:26466/ready


In [ ]:
!curl -sS http://localhost:26466/health


Submit the reviewed single-vulnerability quick-start fixture synchronously to `/generate`. The request blocks until the pipeline completes. HTTP 200 returns a `PipelineResult` with `outcome` set to `complete` or `partial`. HTTP 422 returns either a failed `PipelineResult` or a sanitized `workflow_error` envelope when a nominally complete result fails terminal-artifact validation. The server enforces the full terminal-artifact contract; this cell preserves the response for diagnosis and requires a complete result with at least one vulnerability result before continuing.

> **Note:** The first request can take longer because the workflow must download and extract the root filesystem for the example container, `nvcr.io/nvidia/morpheus/morpheus:25.06-runtime`. Subsequent requests reuse the extracted rootfs.


In [ ]:
import httpx
import json

sync_input = json.loads(Path("data/examples/quick_start_input.json").read_text())
sync_response = httpx.post(
    "http://localhost:26466/generate",
    json=sync_input,
    timeout=3600,
)
try:
    sync_payload = sync_response.json()
except ValueError as exc:
    raise RuntimeError(f"Synchronous API returned non-JSON HTTP {sync_response.status_code}") from exc

sync_result_path = Path("/tmp/deploy_generate_response.json")
sync_result_path.write_text(json.dumps(sync_payload, indent=2) + "\n")
print(sync_result_path.read_text()[:12000])
sync_response.raise_for_status()
if not isinstance(sync_payload, dict):
    raise RuntimeError(f"Synchronous API returned {type(sync_payload).__name__} JSON, expected an object.")
if sync_payload.get("outcome") != "complete":
    raise RuntimeError(f"Synchronous pipeline outcome was {sync_payload.get('outcome')!r}, not 'complete'.")
sync_results = sync_payload.get("results")
if not isinstance(sync_results, list):
    raise RuntimeError("Synchronous pipeline response is missing the results list.")
print(f"Validated {len(sync_results)} successful vulnerability result(s).")


For multi-CVE requests, use the async endpoints by default. The next cells load `pipeline_input.json`, whose vulnerabilities exactly match the reviewed Morpheus eval dataset, submit once, poll to an explicit terminal state, reject `failure` and `interrupted`, and inspect `output.outcome` on terminal `success`. The workflow contract preserves `outcome:partial` output as a completed job; this demonstration still applies complete-result assertions because its fixture expects every vulnerability to succeed.


In [ ]:
async_input = json.loads(Path("data/examples/pipeline_input.json").read_text())
async_submit_response = httpx.post(
    "http://localhost:26466/v1/workflow/async",
    json=async_input,
    timeout=30,
)
try:
    submitted = async_submit_response.json()
except ValueError as exc:
    raise RuntimeError(f"Async submission returned non-JSON HTTP {async_submit_response.status_code}") from exc
Path("/tmp/deploy_async_submit.json").write_text(json.dumps(submitted, indent=2) + "\n")
if isinstance(submitted, dict) and submitted.get("code") == "workflow_error":
    raise RuntimeError(f"Async submission workflow_error: {submitted}")
async_submit_response.raise_for_status()
if not isinstance(submitted, dict):
    raise RuntimeError(f"Async submission returned {type(submitted).__name__} JSON, expected an object.")
job_id = submitted.get("job_id")
if not job_id:
    raise RuntimeError(f"Async submission did not return job_id: {submitted}")
print(json.dumps(submitted, indent=2))


In [ ]:
import time

terminal_states = {"success", "failure", "interrupted"}
deadline = time.monotonic() + 7200
status_url = f"http://localhost:26466/v1/workflow/async/job/{job_id}"
while time.monotonic() < deadline:
    status_response = httpx.get(status_url, timeout=30)
    status_response.raise_for_status()
    try:
        status_payload = status_response.json()
    except ValueError as exc:
        raise RuntimeError(f"Async status returned non-JSON HTTP {status_response.status_code}") from exc
    Path("/tmp/deploy_async_status.json").write_text(json.dumps(status_payload, indent=2) + "\n")
    if not isinstance(status_payload, dict):
        raise RuntimeError(f"Async status returned {type(status_payload).__name__} JSON, expected an object.")
    job_status = status_payload.get("status")
    print(f"status={job_status} updated_at={status_payload.get('updated_at')}")
    if job_status in terminal_states:
        break
    time.sleep(5)
else:
    raise TimeoutError(f"Async job {job_id} did not reach a terminal state within 7200 seconds")

if job_status != "success":
    raise RuntimeError(f"Async job ended in {job_status}: {status_payload.get('error')}")
async_result = status_payload.get("output")
if not isinstance(async_result, dict):
    raise RuntimeError(f"Async job {job_id} succeeded without an object output.")
if async_result.get("outcome") != "complete":
    raise RuntimeError(f"Async pipeline outcome was {async_result.get('outcome')!r}, not 'complete'.")
async_results = async_result.get("results")
if not isinstance(async_results, list):
    raise RuntimeError("Async pipeline response is missing the results list.")
if not async_results:
    raise RuntimeError("Async pipeline response contains no vulnerability results.")
print(f"Validated terminal success for async job {job_id} with {len(async_results)} result(s).")


## Inspect Outputs

The orchestrator writes agent reports and final VEX JSON under `output_dir`, which defaults to `outputs` unless overridden by the input JSON. Inspect the canonical inventory through `GET /reports` so current filenames and URLs come from the service rather than being reconstructed locally.

> **Note:** Relative `output_dir` values are resolved from the repository root, so the default results are stored within the cloned repository at `<repo>/outputs`. The report API returns the exact discoverable paths for those artifacts.


In [ ]:
reports_response = httpx.get("http://localhost:26466/reports", timeout=30)
reports_response.raise_for_status()
report_inventory = reports_response.json()
for group in report_inventory.get("groups", []):
    print(f"group={group['group']}")
    for report_type, entries in group.get("reports", {}).items():
        print(f"  {report_type}: {len(entries)}")
        for entry in entries[:10]:
            print(f"   - {entry['filename']} -> {entry['url']}")


A final VEX JSON record includes the CVE/GHSA identifier, container, VEX status and justification, exploitability verdict, confidence, risk level, and paths to the generated reports.


In [ ]:
vex_entries = [
    entry
    for group in report_inventory.get("groups", [])
    for entry in group.get("reports", {}).get("vex_results", [])
    if entry.get("json_url")
]
for entry in vex_entries[:3]:
    result_response = httpx.get(f"http://localhost:26466{entry['json_url']}", timeout=30)
    result_response.raise_for_status()
    print(f"{entry['filename']}")
    print(json.dumps(result_response.json(), indent=2)[:4000])


## API Reference

With the shipped NAT 1.8 configuration, `nat serve` exposes:

| Method and path | Purpose |
|---|---|
| `POST /generate` | Synchronous run; HTTP 200 returns a `complete` or `partial` `PipelineResult`; HTTP 422 returns either a failed `PipelineResult` or a sanitized `workflow_error` envelope for an unusable nominally complete result |
| `POST /v1/workflow/async` | Submit a background job and receive a `job_id` |
| `GET /v1/workflow/async/job/{job_id}` | Poll async job status/result; terminal states are `success`, `failure`, and `interrupted` |
| `GET /docs` | Swagger UI / OpenAPI docs |
| `GET /health` | NAT HTTP process liveness check |
| `GET /ready` | Readiness check requiring the Dask scheduler and at least one worker |


## Next Steps

Open `deploy/2_Customize_CVE.ipynb` to create custom `PipelineInput` files, analyze multiple CVEs, use pre-extracted rootfs paths, change model settings, and tune concurrency/rate limiting.


## Troubleshooting

- If Docker Compose starts but `/ready` fails, inspect the API log with `bash deploy/compose.sh logs --tail=100 vuln-analysis-api`.
- If API calls fail, re-check `NVIDIA_API_KEY`, `TAVILY_API_KEY`, and `NV_BASE_URL`.
- If a run is too aggressive for your inference endpoint, set `workflow.max_concurrency` or `workflow.llm_max_rate` in a custom config as shown in the customization notebook.
- If you want a clean rerun, delete the relevant Markdown/JSON files under `outputs` or set `"cache": false` in the input JSON.
